In [0]:
# ===========================================
# INSPEKTOR BUDŻET — Notebook 02: Silver
# Cel: Oczyszczenie i przygotowanie danych
# ===========================================

# === MAPOWANIE DOSTAWCÓW NA NIP ===
# Dodaj tutaj nowych dostawców w formacie: "nazwa": NIP
DOSTAWCY_NIP = {
    "rowkop": 5512233444,
    # Przykład: "firma_xyz": 1234567890,
}

dbutils.widgets.text("projekt", "inspektor_budzet")
dbutils.widgets.text("dostawca", "rowkop")

projekt = dbutils.widgets.get("projekt")
dostawca = dbutils.widgets.get("dostawca")

# Widget z NIP automatycznie ustawiany na podstawie dostawcy
nip_domyslny = str(DOSTAWCY_NIP.get(dostawca.lower(), ""))
dbutils.widgets.text("nip_dostawcy", nip_domyslny)
nip = int(dbutils.widgets.get("nip_dostawcy"))

catalog = projekt
schema = dostawca
sciezka_dostawcy = f"/Volumes/{catalog}/{schema}"

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

print(f"Notebook 02 — Silver 🥈")
print(f"Dostawca: {dostawca} ✅")
print(f"NIP:      {nip}")

In [0]:
# TWORZENIE TABELI Z METADANYMI
from datetime import datetime
from pyspark.sql import functions as F

# Moment uruchomienia — zapisujemy raz i używamy w całym notebooku
run_timestamp = datetime.now()
source_table = f"{catalog}.{schema}.bronze_usage_internal"
errors = []  # tutaj będziemy zbierać błędy

# Tworzymy schemat ops jeśli nie istnieje
spark.sql("CREATE SCHEMA IF NOT EXISTS inspektor_budzet.ops")

print(f"Run timestamp: {run_timestamp}")
print(f"Source table:  {source_table}")
print(f"Schemat ops: gotowy")

In [0]:
# Wczytujemy surowe dane z tabeli bronze
df_bronze = spark.table(f"{catalog}.{schema}.bronze_usage_internal")

# Filtrujemy tylko wiersze naszego dostawcy (po NIP z widgetu)
df_rowkop = df_bronze.filter(df_bronze.NIP_DOSTAWCY == nip)

print(f"Wszystkich wierszy w bronze: {df_bronze.count()}")
print(f"Wierszy dla dostawcy {dostawca} (NIP {nip}): {df_rowkop.count()}")
display(df_rowkop)

In [0]:
# --- SPRAWDZENIA JAKOŚCI DANYCH ---

# Check 1: Czy tabela bronze nie jest pusta
if df_bronze.count() == 0:
    errors.append(("row_count_bronze", "Tabela bronze jest pusta"))

# Check 2: Czy są wiersze dla dostawcy
if df_rowkop.count() == 0:
    errors.append(("row_count_dostawca", f"Brak wierszy dla dostawcy {dostawca}"))

# Check 3: Null check na ILOSC
nulls_ilosc = df_rowkop.filter(F.col("ILOSC").isNull()).count()
if nulls_ilosc > 0:
    errors.append(("null_check_ILOSC", f"Kolumna ILOSC zawiera {nulls_ilosc} wartości pustych"))

# Check 4: Ujemne wartości w ILOSC
negative_ilosc = df_rowkop.filter(F.col("ILOSC") <= 0).count()
if negative_ilosc > 0:
    errors.append(("negative_check_ILOSC", f"Kolumna ILOSC zawiera {negative_ilosc} wartości <= 0"))

# Check 5: Czy są TYLKO oczekiwane typy robót (dwukierunkowe sprawdzenie)
expected_types = ["WYKOP_ROWU", "PRACA_OPERATORA"]
actual_types = [row.TYP_ROBOTY for row in df_rowkop.select("TYP_ROBOTY").distinct().collect()]

# Check 6 Sprawdź czy brakuje oczekiwanych typów
for t in expected_types:
    if t not in actual_types:
        errors.append(("missing_typ_roboty", f"Brak oczekiwanego typu roboty: {t}"))

# Check 7 Sprawdź czy są niespodziewane typy
for t in actual_types:
    if t not in expected_types:
        errors.append(("unexpected_typ_roboty", f"Niespodziewany typ roboty: {t}"))

# Check 8: Czy wszystkie wiersze mają wypełnioną nazwę dostawcy
nulls_dostawca = df_bronze.filter(
    F.col("DOSTAWCA").isNull() | (F.col("DOSTAWCA") == "")
).count()
if nulls_dostawca > 0:
    errors.append(("null_check_DOSTAWCA", f"Kolumna DOSTAWCA zawiera {nulls_dostawca} pustych wartości"))

# Podsumowanie
if len(errors) == 0:
    print("Wszystkie sprawdzenia OK ✅")
else:
    print(f"Znaleziono {len(errors)} błędów:")
    for e in errors:
        print(f"  ❌ {e[0]}: {e[1]}")

In [0]:
from pyspark.sql import functions as F

# Sumujemy metry wykopu dla RowKop
df_wykop = (df_rowkop
    .filter(F.col("TYP_ROBOTY") == "WYKOP_ROWU")
    .agg(F.sum("ILOSC").alias("erp_ilosc"))
    .withColumn("position_id", F.lit("WYKOP_ROWU"))
    .withColumn("jednostka", F.lit("mb"))
)

display(df_wykop)

In [0]:
df_operator = (df_rowkop
    .filter(F.col("TYP_ROBOTY") == "PRACA_OPERATORA")
    .agg(F.sum("ILOSC").alias("erp_ilosc"))
    .withColumn("position_id", F.lit("PRACA_OPERATORA"))
    .withColumn("jednostka", F.lit("godz."))
)

display(df_operator)

In [0]:
df_trudny_grunt = (df_rowkop
    .filter(F.col("TYP_ROBOTY") == "WYKOP_ROWU")
    .filter(
        F.col("KATEGORIA_GRUNTU").contains("II") | 
        F.col("KATEGORIA_GRUNTU").contains("III")
    )
    .select("DATA", "LOKALIZACJA", "KATEGORIA_GRUNTU", "PROTOKOL_NR")
)

print(f"Dni z potwierdzonym trudnym gruntem: {df_trudny_grunt.count()}")
display(df_trudny_grunt)

In [0]:
# Zliczamy potwierdzone dni trudnego gruntu
ilosc_trudny_grunt = df_trudny_grunt.count()

# Tworzymy wiersze dla ryczałtu i dodatku
df_ryczalt = spark.createDataFrame([
    (1, "RYCZALT", "m-c")
], ["erp_ilosc", "position_id", "jednostka"])

df_dodatek = spark.createDataFrame([
    (ilosc_trudny_grunt, "DODATEK_TRUDNY_GRUNT", "dzień")
], ["erp_ilosc", "position_id", "jednostka"])

# Łączymy wszystkie 4 pozycje w jedną tabelę
df_silver_summary = (df_wykop
    .union(df_operator)
    .union(df_ryczalt)
    .union(df_dodatek)
)

display(df_silver_summary)

In [0]:
# Zapisujemy tabelę główną
df_silver_summary.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.silver_summary")
print(f"Zapisano: {catalog}.{schema}.silver_summary ✅")

# Zapisujemy szczegółowe dni trudnego gruntu
df_trudny_grunt.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.silver_trudny_grunt_dni")
print(f"Zapisano: {catalog}.{schema}.silver_trudny_grunt_dni ✅")

In [0]:
# Sprawdzam czy tabela silver_summary istnieje i wyświetlam zawartość
df_check = spark.table(f"{catalog}.{schema}.silver_summary")
print(f"Tabela {catalog}.{schema}.silver_summary istnieje ✅")
print(f"Liczba wierszy: {df_check.count()}")
print("\nZawartość tabeli:")
display(df_check)

In [0]:
# --- ZAPIS METADANYCH DO TABEL ---

rows_bronze = df_bronze.count()
rows_dostawca = df_rowkop.count()
status = "ERROR" if len(errors) > 0 else "OK"

df_log = spark.createDataFrame(
    [(run_timestamp, "inspektor_budzet_02_silver", dostawca, source_table, rows_bronze, rows_dostawca, status)],
    ["run_timestamp", "notebook", "dostawca", "source_table", "rows_bronze", "rows_dostawca", "status"]
)
df_log.write.mode("append").saveAsTable("inspektor_budzet.ops.pipeline_log")
print(f"pipeline_log: zapisano 1 wiersz ✅")

if len(errors) > 0:
    df_errors = spark.createDataFrame(
        [(run_timestamp, dostawca, e[0], e[1]) for e in errors],
        ["run_timestamp", "dostawca", "check_name", "opis_bledu"]
    )
    df_errors.write.mode("append").saveAsTable("inspektor_budzet.ops.pipeline_errors")
    print(f"pipeline_errors: zapisano {len(errors)} błędów ✅")
else:
    print("Brak błędów — pipeline_errors bez zmian ✅")